In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

# path setup
try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

# load data dari 01b
df = pd.read_parquet(data_processed / "makassar_features_spatial.parquet")

print("DATA:", df.shape)
df.head()

DATA: (5004844, 14)


,grid_id,date,rainfall,extreme_rain,flood_label,month,rain_lag_1,rain_lag_3,rain_lag_7,rain_lag_14,rain_lag_30,rain_7d_sum,rain_30d_sum,spatial_factor
0,227,2018-03-02,4.044750,0,0,3,19.251960,12.142780,20.213298,16.921814,20.146016,111.850201,551.234117,0.949816
1,227,2018-03-03,11.212212,0,0,3,4.044750,19.746014,17.050726,12.439595,11.562289,106.011687,550.884040,0.949816
2,227,2018-03-04,13.933786,0,0,3,11.212212,19.251960,30.926139,13.286527,8.896039,89.019334,555.921787,0.949816
3,227,2018-03-05,21.319624,0,0,3,13.933786,4.044750,8.687832,20.625382,24.577620,101.651125,552.663790,0.949816
4,227,2018-03-06,19.203036,0,0,3,21.319624,11.212212,12.142780,7.841983,32.136472,108.711380,539.730354,0.949816


In [12]:
np.random.seed(42)

# noise harian
df["rain_noise"] = np.random.normal(0, 5, len(df))

# rainfall dengan noise
df["rainfall_real"] = df["rainfall"] + df["rain_noise"]

# clip agar tidak negatif
df["rainfall_real"] = df["rainfall_real"].clip(lower=0)

print("NOISE ADDED")

NOISE ADDED


In [13]:
df = df.sort_values(["grid_id", "date"])

lags = [1, 3, 7, 14]

for lag in lags:
    df[f"rain_real_lag_{lag}"] = df.groupby("grid_id")["rainfall_real"].shift(lag)

df["rain_real_7d_sum"] = (
    df.groupby("grid_id")["rainfall_real"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

df["rain_real_30d_sum"] = (
    df.groupby("grid_id")["rainfall_real"]
    .rolling(30)
    .sum()
    .reset_index(level=0, drop=True)
)

In [14]:
# parameter (bisa tuning nanti)
THRESH_7D = 250
THRESH_1D = 40

df["flood_label_v2"] = (
    (df["rain_real_7d_sum"] > THRESH_7D) &
    (df["rain_real_lag_1"] > THRESH_1D)
).astype(int)

In [15]:
import numpy as np

np.random.seed(42)

# weighted combination (tidak linear)
score = (
    0.35 * df["rain_real_7d_sum"] +
    0.25 * df["rain_real_lag_1"] +
    0.15 * df["rain_real_lag_3"] +
    0.25 * (df["spatial_factor"] * 100) +
    np.random.normal(0, 25, len(df))  # noise besar
)

# sigmoid → probabilitas
prob = 1 / (1 + np.exp(-0.01 * (score - 220)))

# sampling stochastic
df["flood_label_v2"] = (np.random.rand(len(df)) < prob).astype(int)

In [16]:
# probabilistic flip (real world tidak deterministik)
flip_prob = 0.05

random_flip = np.random.rand(len(df)) < flip_prob

df.loc[random_flip, "flood_label_v2"] = 1 - df.loc[random_flip, "flood_label_v2"]

print("RANDOMNESS ADDED")

RANDOMNESS ADDED


In [17]:
cols_needed = [
    "grid_id", "date",
    "rainfall_real",
    "rain_real_lag_1",
    "rain_real_lag_3",
    "rain_real_lag_7",
    "rain_real_lag_14",
    "rain_real_7d_sum",
    "rain_real_30d_sum",
    "month",
    "spatial_factor",
    "flood_label_v2"
]

df_model = df[cols_needed].dropna()

print("FINAL SHAPE:", df_model.shape)

FINAL SHAPE: (4922658, 12)


In [18]:
print("\n=== FLOOD RATIO ===")
print(df_model["flood_label_v2"].mean())

print("\n=== MONTHLY DISTRIBUTION ===")
print(df_model.groupby("month")["flood_label_v2"].mean())


=== FLOOD RATIO ===
0.23568243010178647

=== MONTHLY DISTRIBUTION ===
month
1     0.235524
2     0.235675
3     0.235639
4     0.234604
5     0.235258
6     0.235639
7     0.236338
8     0.235124
9     0.236763
10    0.235750
11    0.235236
12    0.236581
Name: flood_label_v2, dtype: float64


In [19]:
df_model.sample(10, random_state=42)

,grid_id,date,rainfall_real,rain_real_lag_1,rain_real_lag_3,rain_real_lag_7,rain_real_lag_14,rain_real_7d_sum,rain_real_30d_sum,month,spatial_factor,flood_label_v2
3300387,21107,2022-04-09,14.592141,10.039796,22.878707,1.795374,8.265284,126.771245,564.676879,4,0.852692,0
2487219,20158,2020-01-22,1.649071,6.235445,15.234982,39.974950,74.855904,97.909462,665.055094,1,1.034142,0
215973,16343,2019-08-05,6.107275,23.768606,10.747385,16.886735,3.890044,84.567135,486.911689,8,0.927201,1
3173879,20978,2019-03-14,19.878551,9.429986,20.029033,23.905832,12.680369,115.108766,616.643615,3,0.828834,0
574395,17054,2019-05-21,24.070192,17.051678,12.339238,32.147636,15.513318,138.229109,675.453757,5,1.095159,0
2066646,19497,2019-05-02,0.000000,2.256124,16.837948,32.713712,20.843440,136.729798,482.079233,5,1.046771,1
1648054,18893,2019-03-13,14.442117,48.615964,10.637631,21.147280,18.569066,144.375913,557.003189,3,1.010276,0
1055841,17871,2022-05-19,24.393076,11.444386,8.703356,19.412354,10.693691,132.968462,480.996135,5,0.868928,0
794086,17502,2021-04-27,12.086956,22.872718,6.715709,22.527906,8.697893,96.514045,473.779316,4,0.833400,0
1536848,18676,2019-05-04,5.926819,9.979267,31.676550,4.141692,0.000000,111.773841,563.346662,5,0.949324,0


In [20]:
out_path = data_processed / "makassar_final_dataset.parquet"

df_model.to_parquet(out_path, index=False)

print("✔ FINAL DATASET SAVED:", out_path)

✔ FINAL DATASET SAVED: /workspaces/flood-ml-research/data/processed/makassar_final_dataset.parquet


In [21]:
print("Flood ratio:", df_model["flood_label_v2"].mean())

Flood ratio: 0.23568243010178647


In [22]:
df_model.to_parquet(
    data_processed / "makassar_final_dataset.parquet",
    index=False
)